# Supervisor-Worker model in LangGraph
Companion notebook for the **Supervisor-Worker model in LangGraph** section of Chapter 4 (Supervisor Node, SEND and COMMAND statements, Worker Nodes, Using Subgraphs as Worker Nodes).

Three small, independent demos, one per sub-section. Notebook 09 assembles the full capstone agent that combines all of these ideas together.

## Supervisor Node
A Supervisor node does not do the heavy work itself -- it inspects the state and returns a `Command`, which tells LangGraph which node to jump to next, bypassing the normal edges.

In [1]:
from typing import TypedDict
from langgraph.graph import StateGraph, START, END
from langgraph.types import Command


class SupervisorState(TypedDict):
    query: str
    candidate_urls: list[str]
    final_report: str

In [2]:
def prepare_query(state: SupervisorState) -> dict:
    """Simplified stand-in: cleans the query AND fetches sources in one step.
    (Notebook 09 shows the fully separated prepare / search / fetch pipeline.)"""
    cleaned = state["query"].strip()
    return {"query": cleaned, "candidate_urls": [f"https://example.com/{cleaned}"]}


def summarize_results(state: SupervisorState) -> dict:
    return {"final_report": f"Report based on {len(state['candidate_urls'])} source(s)"}

This is the supervisor exactly as shown in the book -- it inspects `candidate_urls` and `final_report` and decides what runs next:

In [3]:
def supervisor_node(state: SupervisorState):
    if not state.get("candidate_urls"):
        return Command(goto="prepare_query")

    if not state.get("final_report"):
        return Command(goto="summarize_results")

    return Command(goto="__end__")

In [4]:
builder = StateGraph(SupervisorState)
builder.add_node("supervisor", supervisor_node)
builder.add_node("prepare_query", prepare_query)
builder.add_node("summarize_results", summarize_results)

# Note there is no static edge out of "supervisor" -- Command handles that at runtime.
builder.add_edge(START, "supervisor")
builder.add_edge("prepare_query", "supervisor")
builder.add_edge("summarize_results", "supervisor")

app = builder.compile()
result = app.invoke({"query": "  LangGraph architecture  ", "candidate_urls": [], "final_report": ""})
print(result)

{'query': 'LangGraph architecture', 'candidate_urls': ['https://example.com/LangGraph architecture'], 'final_report': 'Report based on 1 source(s)'}


## SEND and COMMAND statements
`Command` (used above) jumps execution to a specific node after applying a state update. `Send`, shown next, does the opposite kind of thing: it *multiplies* execution, spawning one parallel worker invocation per item, each with its own slice of state. The graph keeps moving forward; the results are merged back using reducers (see notebook 05).

## Worker Nodes
Below, `send_wikipedia_urls` returns one `Send(...)` per Wikipedia URL, dynamically spawning a `fetch_page_wiki` worker for each -- this is *Dynamic Spawning*.

In [5]:
import operator
from typing import Annotated
from langgraph.types import Send


class FanOutState(TypedDict, total=False):
    candidate_urls: list[str]
    current_url: str
    fetched: Annotated[list[str], operator.add]


def send_wikipedia_urls(state: FanOutState):
    sends = []
    for url in state["candidate_urls"]:
        if "wikipedia.org" in url:
            sends.append(
                Send("fetch_page_wiki", {"current_url": url})
            )
    return sends

In [6]:
def fetch_page_wiki(state: FanOutState) -> dict:
    # In a real agent this would fetch and clean the page content.
    return {"fetched": [f"content-of({state['current_url']})"]}


fanout_builder = StateGraph(FanOutState)
fanout_builder.add_node("dispatch", lambda state: {})
fanout_builder.add_node("fetch_page_wiki", fetch_page_wiki)
fanout_builder.add_edge(START, "dispatch")
fanout_builder.add_conditional_edges("dispatch", send_wikipedia_urls, ["fetch_page_wiki"])
fanout_builder.add_edge("fetch_page_wiki", END)

fanout_app = fanout_builder.compile()
result = fanout_app.invoke({
    "candidate_urls": [
        "https://en.wikipedia.org/wiki/LangGraph",
        "https://en.wikipedia.org/wiki/Graph_theory",
        "https://example.com/not-wiki",
    ],
    "fetched": [],
})
print(result["fetched"])

['content-of(https://en.wikipedia.org/wiki/LangGraph)', 'content-of(https://en.wikipedia.org/wiki/Graph_theory)']


## Using Subgraphs as Worker Nodes
A subgraph is a complete, compiled LangGraph workflow used as a single node inside a parent graph. Here, `wiki_search` and `tavily_search` run in parallel *inside* the subgraph, and the compiled subgraph is then added to the parent as one node called `"search"`.

In [7]:
class SearchState(TypedDict):
    query: str
    candidate_urls: Annotated[list[str], operator.add]


def wiki_search(state: SearchState) -> dict:
    return {"candidate_urls": [f"https://en.wikipedia.org/wiki/{state['query']}"]}


def tavily_search(state: SearchState) -> dict:
    return {"candidate_urls": [f"https://example.com/search?q={state['query']}"]}


def build_search_subgraph():
    sg = StateGraph(SearchState)
    sg.add_node("wiki_search", wiki_search)
    sg.add_node("tavily_search", tavily_search)
    sg.add_edge(START, "wiki_search")
    sg.add_edge(START, "tavily_search")
    sg.add_edge("wiki_search", END)
    sg.add_edge("tavily_search", END)
    return sg.compile()


search_graph = build_search_subgraph()

In [8]:
# Use the compiled subgraph as a single node in the parent graph.
parent_builder = StateGraph(SearchState)
parent_builder.add_node("search", search_graph)
parent_builder.add_edge(START, "search")
parent_builder.add_edge("search", END)
parent_app = parent_builder.compile()

result = parent_app.invoke({"query": "LangGraph", "candidate_urls": []})
print(result["candidate_urls"])

['https://example.com/search?q=LangGraph', 'https://en.wikipedia.org/wiki/LangGraph']
